## Gene essentiality analysis

Gene ids are matched against the experimental dataset's `BSUxxxxx` locus tags after light
normalization (uppercase, strip stray underscores). Ids that aren't locus tags at all - BsubCyc
`BG` ids, GECKO enzyme-usage pseudo-genes, gap-filling placeholders - are expected to not match.

## Apply M9 media

In [1]:
import pandas as pd
from cobra.io import read_sbml_model


def set_culture_media(model, media):
    """Apply the media bounds, closing any `_reverse` import partner.

    GECKO models bypass the medium cap entirely if those partners are left open: ecBSU1 was
    taking up 9.3 mmol/gDW/h of ammonia against an M9 cap of 5.
    """
    for _, row in media.iterrows():
        reaction_id = row['ID']
        if reaction_id not in model.reactions:
            continue
        model.reactions.get_by_id(reaction_id).bounds = (row['LOWER_BOUND'], row['UPPER_BOUND'])

        reverse_id = f"{reaction_id}_reverse"
        if reverse_id in model.reactions:
            model.reactions.get_by_id(reverse_id).bounds = (0.0, 0.0)
    return model


medium = pd.read_csv("../data/datasets/M9_media.csv")

## Metrics helpers

In [2]:
def calculate_metrics(tp, fp, tn, fn, total_genes_model, total_genes_dataset):

    metrics = {}
    
    if (tp + fn) > 0:
        metrics['tpr'] = tp / (tp + fn)
        metrics['recall'] = tp / (tp + fn)  # Same as TPR
    else:
        metrics['tpr'] = 0.0
        metrics['recall'] = 0.0
    
    if (tn + fp) > 0:
        metrics['tnr'] = tn / (tn + fp)
        metrics['specificity'] = tn / (tn + fp)  # Same as TNR
    else:
        metrics['tnr'] = 0.0
        metrics['specificity'] = 0.0
    
    if (fp + tp) > 0:
        metrics['fdr'] = fp / (fp + tp)
    else:
        metrics['fdr'] = 0.0
    
    if (tp + fp) > 0:
        metrics['precision'] = tp / (tp + fp)
    else:
        metrics['precision'] = 0.0
    
    if metrics['precision'] + metrics['recall'] > 0:
        metrics['f1_score'] = 2 * (metrics['precision'] * metrics['recall']) / \
                              (metrics['precision'] + metrics['recall'])
    else:
        metrics['f1_score'] = 0.0
    
    total = tp + fp + tn + fn
    if total > 0:
        metrics['accuracy'] = (tp + tn) / total
    else:
        metrics['accuracy'] = 0.0
    
    denominator = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    if denominator > 0:
        metrics['mcc'] = (tp * tn - fp * fn) / denominator
    else:
        metrics['mcc'] = 0.0
    
    # Coverage: genes in model / genes in the reference dataset
    if total_genes_dataset > 0:
        metrics['coverage'] = total_genes_model / total_genes_dataset
    else:
        metrics['coverage'] = 0.0
    
    return metrics

Parses the dataset's essentiality column into a bool (handles various yes/no encodings).

In [3]:
def parse_essentiality(value):

    if pd.isna(value):
        return False
    
    value_str = str(value).lower().strip()
    
    # Consider as essential
    if value_str in ['yes', 'y', 'true', '1', 'essential']:
        return True
    
    # Everything else (including 'no', empty, etc.) is non-essential
    return False

## Gene essentiality comparison

Knocks out each gene in turn (`find_essential_genes`), matches predictions against the
experimental dataset by normalized locus tag, and reports a confusion matrix per model.

In [4]:
import sys
import os
import numpy as np
import cobra


def normalize_gene_id(gene_id):
    # Locus tags should read "BSUxxxxx"; some models carry a stray underscore
    # ("BSU_xxxxx"). Ids that aren't locus tags (BsubCyc "BG" ids, GECKO enzyme-usage
    # pseudo-genes like "E_Pgi", gap-filling placeholders) can't be normalized this way
    # and are intentionally left to fail matching.
    return gene_id.strip().upper().replace("BSU_", "BSU")


def gene_essentiality_comparison(
            models, dataset_path, locus_column, essential_column,
            outdir, output_name, processes=4
        ):

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset file not found: {dataset_path}")

    df_exp = pd.read_csv(dataset_path)

    if locus_column not in df_exp.columns:
        raise ValueError(f"Dataset must contain '{locus_column}' column")
    if essential_column not in df_exp.columns:
        raise ValueError(f"Dataset must contain '{essential_column}' column")

    experimental_data = {
        row[locus_column]: parse_essentiality(row[essential_column])
        for _, row in df_exp.iterrows()
    }

    exp_essential_count = sum(experimental_data.values())
    print(f"[INFO] Experimental data loaded:")
    print(f"  Total genes: {len(experimental_data)}")
    print(f"  Essential: {exp_essential_count}")
    print(f"  Non-essential: {len(experimental_data) - exp_essential_count}")

    os.makedirs(outdir, exist_ok=True)
    output_file = os.path.join(outdir, f"{output_name}.csv")

    summary_data = []
    for model_path in models:
        model_name = os.path.splitext(os.path.basename(model_path))[0]
        print(f"\n[INFO] Processing model: {model_name}")

        try:
            model = read_sbml_model(model_path)
        except Exception as e:
            print(f"[ERROR] Could not load model, trying json loading {model_path}: {e}", file=sys.stderr)
            model = cobra.io.load_json_model(model_path)

        set_culture_media(model, medium)

        essential_genes_obj = cobra.flux_analysis.find_essential_genes(model, processes=processes)
        predicted_essential_ids = {gene.id for gene in essential_genes_obj}

        # normalized gene id -> original gene id, so predictions can be looked up after matching
        model_genes_map = {normalize_gene_id(gene.id): gene.id for gene in model.genes}

        common_genes = set(experimental_data.keys()) & set(model_genes_map.keys())
        if len(common_genes) == 0:
            print(f"[WARNING] No common genes found between dataset and {model_name}",
                  file=sys.stderr)
            continue

        tp, fp, tn, fn = 0, 0, 0, 0
        for gene_normalized in common_genes:
            exp_essential = experimental_data[gene_normalized]
            pred_essential = model_genes_map[gene_normalized] in predicted_essential_ids

            if exp_essential and pred_essential:
                tp += 1
            elif not exp_essential and pred_essential:
                fp += 1
            elif not exp_essential and not pred_essential:
                tn += 1
            else:
                fn += 1

        metrics = calculate_metrics(tp, fp, tn, fn, len(model.genes), len(experimental_data))
        summary_data.append({
            'model': model_name,
            'genes_in_model': len(model.genes),
            'genes_in_dataset': len(experimental_data),
            'common_genes': len(common_genes),
            'TP': tp,
            'FP': fp,
            'TN': tn,
            'FN': fn,
            'TPR': round(metrics['tpr'], 4),
            'TNR': round(metrics['tnr'], 4),
            'FDR': round(metrics['fdr'], 4),
            'MCC': round(metrics['mcc'], 4),
            'Precision': round(metrics['precision'], 4),
            'Recall': round(metrics['recall'], 4),
            'Specificity': round(metrics['specificity'], 4),
            'F1_Score': round(metrics['f1_score'], 4),
            'Accuracy': round(metrics['accuracy'], 4),
            'Coverage': round(metrics['coverage'], 4)
        })

    df_summary = pd.DataFrame(summary_data)
    df_summary.to_csv(output_file, index=False)

    return {
        'summary': df_summary,
        'output_file': output_file,
    }

In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

def plot_metrics_comparison(df_summary, outdir, output_name):

    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        # Math in labels ($\mu$, h$^{-1}$) otherwise renders in matplotlib's default DejaVu while
        # the surrounding text is Arial, so one axis label ends up in two typefaces.
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Arial',
    'mathtext.it': 'Arial:italic',
    'mathtext.bf': 'Arial:bold',
        'font.size': 11,
        'axes.labelsize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'figure.dpi': 300,
        'savefig.dpi': 800,
        'axes.linewidth': 0.8,
    })

    # Coverage dropped: at ~0.2 it compresses the other metrics on a shared 0-1 scale. FDR and
    # F1 added so every metric defined in the Methods appears. Rows clustered, columns fixed.
    metrics = ['Precision', 'FDR', 'Recall', 'Specificity', 'F1_Score', 'Accuracy', 'MCC']

    df = df_summary.set_index('model')[metrics]
    df.columns = [m.replace('_Score', '') for m in metrics]

    vmin = np.floor(df.values.min() * 10) / 10
    vmax = np.ceil(df.values.max() * 10) / 10

    g = sns.clustermap(
        df,
        cmap='YlGnBu',
        vmin=vmin,
        vmax=vmax,
        annot=True,
        fmt='.2f',
        annot_kws={'fontsize': 9},
        row_cluster=True,
        col_cluster=False,
        dendrogram_ratio=(0.18, 0.0),
        cbar_pos=(1.02, 0.3, 0.03, 0.4),
        linewidths=1.5,
        linecolor='white',
        figsize=(7.5, 0.7 * len(df) + 1.5),
        tree_kws={'linewidths': 1.2},
    )
    g.ax_heatmap.set_ylabel('')
    g.ax_heatmap.set_xlabel('')
    g.ax_heatmap.tick_params(left=False, bottom=False)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), rotation=0)
    g.cax.set_ylabel('Score', fontsize=10)

    plot_path = os.path.join(outdir, f"{output_name}_metrics_comparison.pdf")
    g.savefig(plot_path, dpi=800, bbox_inches='tight')
    plt.close(g.fig)

    return plot_path


## Run

ModelSEED/mergem-edge-case models (`iBsu1103`, `iBsu1103v2`, `iBsu1209`) are
excluded here, same as notebooks 03+.

In [6]:
final_dir = os.path.join("..", "data", "models", "02_final_annotated")
model_names = ["iYO844", "iBsu1147", "eciYO844", "iBB1018", "iBsu1147R", "ecBSU1"]

results=gene_essentiality_comparison(
    models=[os.path.join(final_dir, f"{name}.xml") for name in model_names],
    dataset_path="../data/datasets/Essentiality_dataset.csv",
    locus_column= "genomic_annotation.locus_tag",
    essential_column= "essential",
    outdir= "../results/tables",
    output_name= "GeneEssentiality",
    processes= 4
)


[INFO] Experimental data loaded:
  Total genes: 4332
  Essential: 256
  Non-essential: 4076

[INFO] Processing model: iYO844


Set parameter Username


Set parameter LicenseID to value 2695449


--------------------------------------------


--------------------------------------------


...


[INFO] Processing model: iBB1018



[INFO] Processing model: iBsu1147R


'' is not a valid SBML 'SId'.



[INFO] Processing model: ecBSU1


'' is not a valid SBML 'SId'.


## Plot

In [7]:
plot_metrics_comparison(
    df_summary=pd.read_csv("../results/tables/GeneEssentiality.csv"),
    outdir="../results/figures",
    output_name="Plot_GeneEssentiality"
)

'../results/figures\\Plot_GeneEssentiality_metrics_comparison.pdf'